<a href="https://colab.research.google.com/github/SofiaDomeli/Neural_Network_HS/blob/titanic/RedesNeurais_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <center> 🧊 Rede Neural Titanic 🚢 </center>

por Heny Dorta e Sofia Domingues 😼

### 1. Bibliotecas

In [48]:
import pandas as pd
import numpy as np
import random
from itertools import product
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.initializers import GlorotUniform

### 2. Variáveis e Ambiente

In [49]:
# =======================
# 1. Leitura e tratamento do dataset
# =======================
df = pd.DataFrame(sns.load_dataset('titanic')).drop(columns=['alive'])

# Tratamento de nulos
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Ajuste de tipos
df = df.astype({
    "survived": "int8",
    "pclass": "int8",
    "sibsp": "int8",
    "parch": "int8",
    "age": "float32",
    "fare": "float32"
})

X = df.drop(columns=['survived'], axis=1)
y = np.array(df['survived'])

# Explicitamente separar colunas
categorical_cols = X.select_dtypes(include=['object', 'bool', 'category']).columns
numeric_cols = X.select_dtypes(include=np.number).columns

# =======================
# 2. Split 80/20 com seed=42
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =======================
# 3. Pré-processamento
# =======================
preprocessor = ColumnTransformer(
    transformers=[
        ("categoricas", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
         categorical_cols),
        ("numericas", StandardScaler(), numeric_cols)
    ],
    remainder="drop",  # não deixa passar colunas não tratadas
    verbose_feature_names_out=False
)

# aplica somente no treino, depois transforma teste
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),
                       columns=preprocessor.get_feature_names_out()).astype("float32")
X_test = pd.DataFrame(preprocessor.transform(X_test),
                      columns=preprocessor.get_feature_names_out()).astype("float32")

# codifica resposta
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# =======================
# 4. Função para criar modelo
# =======================
def criar_modelo(neuronios=32, camadas=2, ativacao="sigmoid", otimizador="adam"):
    model = Sequential()
    model.add(Dense(neuronios, input_dim=X_train.shape[1],
                    activation=ativacao, kernel_initializer=GlorotUniform(seed=42)))
    for _ in range(camadas - 1):
        model.add(Dense(neuronios, activation=ativacao,
                        kernel_initializer=GlorotUniform(seed=42)))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer=otimizador, loss="binary_crossentropy", metrics=["accuracy"])
    return model

# =======================
# 5. Espaço de busca
# =======================
param_grid = {
    "neuronios": [8, 16, 32, 64],
    "camadas": [1, 2, 3, 4],
    "ativacao": ["sigmoid", "relu", "tanh"],
    "otimizador": ["adam", "sgd"],
    "epochs": [50, 100]
}

# todas as combinações
todas_combinacoes = list(product(*param_grid.values()))

# Random Search: pega 15 combinações aleatórias
num_testes = 15
combinacoes_teste = random.sample(todas_combinacoes, min(num_testes, len(todas_combinacoes)))

melhor_acc = 0
melhores_params = None
historico = []

# =======================
# 6. Teste das combinações
# =======================
for valores in combinacoes_teste:
    params = dict(zip(param_grid.keys(), valores))

    model = criar_modelo(
        neuronios=params["neuronios"],
        camadas=params["camadas"],
        ativacao=params["ativacao"],
        otimizador=params["otimizador"]
    )

    model.fit(X_train, y_train, epochs=params["epochs"], verbose=0)

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    historico.append((params, acc))

    if acc > melhor_acc:
        melhor_acc = acc
        melhores_params = params

    print(f"Testado: {params} -> Acurácia: {acc:.4f}")

# =======================
# 7. Resultado final
# =======================
print("\n✅ Melhores parâmetros encontrados:")
print(melhores_params)
print(f"📊 Acurácia no teste: {melhor_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testado: {'neuronios': 64, 'camadas': 3, 'ativacao': 'sigmoid', 'otimizador': 'sgd', 'epochs': 100} -> Acurácia: 0.6145


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testado: {'neuronios': 64, 'camadas': 3, 'ativacao': 'relu', 'otimizador': 'adam', 'epochs': 100} -> Acurácia: 0.7933


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

Testado: {'neuronios': 16, 'camadas': 1, 'ativacao': 'tanh', 'otimizador': 'adam', 'epochs': 100} -> Acurácia: 0.7989


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testado: {'neuronios': 32, 'camadas': 3, 'ativacao': 'relu', 'otimizador': 'sgd', 'epochs': 100} -> Acurácia: 0.8212


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Testado: {'neuronios': 32, 'camadas': 1, 'ativacao': 'relu', 'otimizador': 'adam', 'epochs': 100} -> Acurácia: 0.8101

✅ Melhores parâmetros encontrados:
{'neuronios': 64, 'camadas': 1, 'ativacao': 'tanh', 'otimizador': 'sgd', 'epochs': 100}
📊 Acurácia no teste: 0.8268
